# Manage ExaMLOps calculation plugins from a notebook

Author, test, activate, and remove the **provider plugins** behind your project's calculations
(FinOps `cost`/`carbon`, `drift`, `promotion`, `llm_*`, …) — all AST-sandboxed and per-project
(ADR 0074). This notebook is read-only under `/repo/docs/notebooks/`; **File → Save As** into your
own `work/` folder to edit and run it.

The `examlops` package is on the path and `EXAMLOPS_PROVIDERS_DIR` points at the shared store, so
anything you save here is used by the `exa` CLI, the dashboard, and serving.

In [ ]:
import os
from examlops.providers import (
    register_from_source, save_provider, list_project_providers,
    set_active_provider, get_active_provider, read_provider_source, delete_provider,
)
PROJECT = "minio-demo"  # <- your project
print("providers dir:", os.getenv("EXAMLOPS_PROVIDERS_DIR"))
list_project_providers(PROJECT)

## 1. Write a plugin
A provider is one `class X(Provider)` with `compute(inputs) -> dict`. **No imports** — `Provider`,
`ProviderMeta`, and `math` are pre-injected. The sandbox rejects `import`/`eval`/`open`/… before it runs.

In [ ]:
code = '''
class MyCost(Provider):
    name = "my-cost"
    version = "1.0"

    def metadata(self):
        return ProviderMeta(methodology="gpu_hours*0.85 + cpu_hours*0.05", outputs=("cost_usd",))

    def compute(self, inputs):
        return {"cost_usd": inputs.get("gpu_hours", 0) * 0.85 + inputs.get("cpu_hours", 0) * 0.05}
'''

# Test it live in this session (no persistence yet):
register_from_source("cost", "my-cost", code)
from examlops.providers import get_provider
get_provider("cost", config={"provider": "my-cost"}).compute({"gpu_hours": 10, "cpu_hours": 100})

## 2. Persist + activate it
`save_provider` writes it to the shared store; `set_active_provider` makes it the default for
`(project, domain)` — used whenever cost is computed for this project without an explicit `--provider`.

In [ ]:
save_provider("cost", "my-cost", code, project=PROJECT, actor="notebook")
set_active_provider(PROJECT, "cost", "my-cost")
list_project_providers(PROJECT)

## 3. Confirm the platform uses it
FinOps resolves the project's active provider automatically:

In [ ]:
from examlops.finops.cost import estimate_cost_via_provider
estimate_cost_via_provider(10.0, project=PROJECT)  # -> {'cost_usd': ..., 'provider': 'my-cost', ...}

## 4. Housekeeping
The safety gate blocks anything dangerous; and you can read or delete a plugin.

In [ ]:
from examlops.providers import ProviderSecurityError, validate_source
try:
    validate_source("import os")
except ProviderSecurityError as e:
    print("blocked:", e)

print(read_provider_source(PROJECT, "cost", "my-cost")[:80], "...")
# delete_provider(PROJECT, "cost", "my-cost")  # uncomment to remove